# 06 · Evaluation comparison (baseline vs v1 to v6)

All numbers come from saved results on the frozen `nestling-golden-v2` set (104 cases). Code metrics are in `evals/results/*_summary.json`; faithfulness is from the Nebius judge CSVs (`judge-v1`, Qwen3-235B), computed with `evals/judge.py::summary`, which excludes judge errors. v1 to v4 were measured with round-1 code; v5 is the round-2 pipeline. v1 to v5 were run on Claude (Sonnet 5.5 / Haiku 5.5); v6 runs on Nebius open models (DeepSeek-V4-Pro / V4.1-Flash) with the strict Verifier and code-enforced citations, so v6 is not a like-for-like step from v5. The logic is in `evals/compare.py`.

In [1]:
from evals import compare

table = compare.comparison_table()
print(compare.format_table(table))

version                 esc             refuse              route               cite              faith              p95 s            mean Rs             p95 Rs
baseline             100.0%            97.1% X            84.6% X              78.1%            89.3% X             8.57 X              1.305              1.502
v1           100.0% (+0.0%)    99.0% (+1.9%) X    79.8% (-4.8%) X      82.2% (+4.1%)    85.7% (-3.6%) X       7.58 (-0.99)     1.007 (-0.298)     1.513 (+0.011)
v2           100.0% (+0.0%)    99.0% (+0.0%) X    84.6% (+4.8%) X      80.8% (-1.4%)    79.4% (-6.3%) X     8.31 (+0.73) X      0.917 (-0.09)     1.541 (+0.028)
v3           100.0% (+0.0%)    99.0% (+0.0%) X    82.7% (-1.9%) X      78.1% (-2.7%)    76.1% (-3.3%) X    10.55 (+2.24) X     0.951 (+0.034)     1.555 (+0.014)
v4           100.0% (+0.0%)    99.0% (+0.0%) X    81.7% (-1.0%) X     67.1% (-11.0%)     94.1% (+18.0%)    16.91 (+6.36) X      1.971 (+1.02)    4.625 (+3.07) X
v5           100.0% (+0.0%)     10

## What the table says

- **Safety held throughout:** escalation recall is 100% in every run, and v5 reaches refusal 100% with zero wrong refusals.
- **Routing:** best at v5 (97.1%, +15.4 over v4, passes the 90% bar); v1 to v4 never reached it.
- **Faithfulness:** baseline 89.3%, dips to 76.1% at v3, peaks at 94.1% with the Verifier (v4), and ends at 84.1% in v5, below the 90% bar.
- **Citation match** (no bar of its own) ends at 75.3%, below the baseline's 78.1%.
- **Still failing at v5:** faithfulness (84.1%), p95 latency (15.48 s vs < 8 s) and p95 cost (3.688 INR vs < 2 INR). Mean cost (1.722 INR) passes.
- The plan's per-lever predictions mostly missed: routing +4.8 (predicted +15 to 20), retrieval -3.3 faithfulness (predicted +8 to 10), Verifier +18 (predicted +5) at a large latency and cost price.

### v6 (open models, strict Verifier, code-enforced citations)

- **Safety held:** escalation recall 100%, refusal 100%, routing 95.2% (passes the 90% bar).
- **Faithfulness did not recover:** 80.3% (66 judged, 3 judge errors) against v5 84.1% and v4 94.1%; still below the 90% bar. Ten cases are faithful in v5 and unfaithful in v6.
- **Citation match fell to 57.5%** (v5 75.3%, baseline 78.1%). Wrong-source-family misses rose from 18 to 28. Citations are now all real (none invalid or unretrieved), but DeepSeek cites the broad antenatal guide where the golden set wants WHO-ANC, USER-POLICY or others.
- **Cost fell:** mean 0.754 INR and p95 1.462 INR pass the bars; p95 latency 14.53 s still misses the 8 s bar.


## Faithfulness regressions, v4 to v5

Cases the judge found faithful in v4 and unfaithful in v5, with the unsupported claims.

In [2]:
regressions = compare.faithfulness_regressions("v4", "v5")
print(len(regressions), "regressions")
for r in regressions:
    print(f"\n{r['case_id']} ({r['agent']}, {r['scenario_type']}): {r['question'][:90]}")
    print("   unsupported:", r["unsupported"][:300])

9 regressions

B06 (care, edge): Please explain pregnancy danger signs in simple Tamil mixed with English.
   unsupported: If you are 32 weeks pregnant, you should know these danger signs.

B13 (care, edge): Our baby missed the six-week vaccines because of illness. Should we restart all vaccines?
   unsupported: Bring your baby's vaccination card so they can see what has been given.

E02 (care, edge): Mild headache, otherwise fine, week 14
   unsupported: A mild headache with no other symptoms is very common in pregnancy.

F05 (ready, happy): How can I make a hospital emergency plan before my wife goes into labour?
   unsupported: Photo ID and insurance details are needed

F08 (money, happy): Will my employer group insurance cover my spouse delivery and newborn NICU?
   unsupported: Please upload the policy document, or ask your HR team, insurer or TPA directly. | Good things to ask: is maternity covered, is there a waiting period, what is the sum insured, and is the baby covered from 

## Faithfulness regressions, v5 to v6

Cases faithful in v5 and unfaithful in v6.

In [3]:
r6 = compare.faithfulness_regressions("v5", "v6")
print(len(r6), "regressions")
for r in r6:
    print(f"{r['case_id']} ({r['agent']}): {r['unsupported'][:160]}")
for v in ("v5", "v6"):
    print(v, {m: len(i) for m, i in compare.failure_modes(v).items()}, "citation misses:", len(compare.citation_misses(v)))
print("v6 judge errors (not scored):", [r["case_id"] for r in compare.load_judge("v6") if r["faithful"] is None])

10 regressions
B14 (care): It's a key part of protecting your baby's health. | A visit to a paediatrician is a broader health check-up. | It's a chance for a doctor to examine your baby's
E06 (care): Many mothers have a drink before they know they are pregnant.
F13 (ready): A C-section is major surgery, so her rest is crucial. | If she is breastfeeding, you can be the one to lift the baby from the cot, change the nappy, and settle 
F15 (care): The first bath should be delayed for at least 24 hours to protect the baby | The first bath should be delayed for at least 24 hours after birth
H03 (care): feeling tired in the first trimester is very common
H18 (money): You can typically add your newborn to your existing health insurance policy.
M11 (care): the shape of the bump can tell you the baby's sex
M16 (ready): Your body needs rest to recover from surgery, so you should not be lifting anything heavier than your baby. | Plan for help with household tasks like cooking, c
M20 (care): giving

## Failure modes

Failing cases grouped by category (a case can be in more than one). Definitions are in `compare.failure_modes`.

In [4]:
for v in ("baseline", "v5", "v6"):
    print(v)
    for mode, ids in compare.failure_modes(v).items():
        print(f"  {mode:22} {len(ids):3}  {', '.join(ids[:14])}{' ...' if len(ids) > 14 else ''}")

baseline
  routing boundary        16  B16, E10, E11, F01, F02, F03, F13, F22, M04, M06, M09, M11, M19, M20 ...
  refusal calibration      3  B11, B18, E05
  wrong source family     15  B11, F01, F06, F08, F14, F21, H06, M03, M04, M10, M12, M17, M18, M19 ...
  unsupported claim        8  B05, E02, F13, F16, F19, H12, H13, M07
  citation mismatch        1  F17
v5
  routing boundary         3  M03, M16, M23
  refusal calibration      0  
  wrong source family     18  B11, B13, B14, F01, F06, F08, F21, H06, H18, M03, M04, M07, M08, M10 ...
  unsupported claim       11  B05, B06, B13, E02, F05, F08, F10, F14, H08, H16, M09
  citation mismatch       10  B17, F04, F05, F10, F20, H02, H13, M02, M05, M24
v6
  routing boundary         5  E03, H16, M03, M16, M23
  refusal calibration      0  
  wrong source family     28  B08, B09, B11, B13, B14, E09, F01, F02, F06, F08, F21, H01, H06, H07 ...
  unsupported claim       13  B14, E06, F10, F13, F14, F15, H03, H18, M06, M11, M16, M20, M24
  citatio

In [5]:
print("v5 citation-match misses:", len(compare.citation_misses("v5")))
for m in compare.top_failure_modes("v5"):
    print(f"top mode: {m['mode']} ({m['count']})")

v5 citation-match misses: 18
top mode: wrong source family (18)
top mode: unsupported claim (11)
top mode: citation mismatch (10)


## Top 3 failure modes at v5

1. **Wrong source family (18 cases).** The answer cites sources, but not the required family (for example the antenatal guide for a postnatal or immunisation question, or a general scheme instead of the user's policy). Largest single group and the main reason citation match is 75.3%.
2. **Unsupported claim (11 cases).** The judge finds statements the cited passages do not carry (nine are new since v4). v5 also changed the Verifier to a single rewrite; this analysis does not isolate that as the cause.
3. **Citation mismatch (10 cases).** The required source is missed with nothing cited, or a citation was never retrieved.

Routing boundary misses fell to 3 (M03, M16, M23) and there are no refusal misses. The full lists, with the 18 citation misses and 9 regressions, are in `evals/results/phase3_failure_analysis.json`.

## Live run (off)

This notebook reads saved results only.

In [6]:
RUN_LIVE = False

if RUN_LIVE:
    from dotenv import find_dotenv, load_dotenv

    load_dotenv(find_dotenv(usecwd=True))
    from nestling import graph

    graph.run_experiment("v6")
else:
    print("RUN_LIVE = False: showing saved results only")

RUN_LIVE = False: showing saved results only
